# Companion Notebook — Chapter 08: Data Preparation & Exploratory Analysis
**Artificial Intelligence Learning Hub** — *Institut Teknologi Sumatera (ITERA)*  
**Penulis**: Aji Pamungkas Tri Nurcahyo  

Notebook ini memandu alur kerja penyiapan data (*data preparation*) dan analisis data eksploratif (*Exploratory Data Analysis* - EDA) menggunakan dataset nyata telemetri turbin generator industri (`telemetry_raw.csv`).

### Agenda Praktik:
1. **Inspeksi Awal Data**: Struktur baris-kolom, tipe data, dan inkonsistensi teks.
2. **Penanganan Missing Values**: Analisis mekanisme kekosongan dan imputasi terkelompok (*group median*).
3. **Pembersihan Duplikasi**: Identifikasi duplikat identik vs subset observasi sensor berulang.
4. **Deteksi dan Penanganan Outlier**: Metode Tukey IQR, Z-Score, dan pertimbangan domain engineering (*capping/Winsorizing*).
5. **Transformasi Fitur Numerik**: Normalisasi Min-Max, Standarisasi Z-score, dan Transformasi Logaritma untuk mereduksi *skewness*.
6. **Encoding Fitur Kategorikal**: Ordinal Encoding vs One-Hot Encoding.
7. **Analisis Korelasi & Multivariat**: Matriks korelasi Pearson dan visualisasi sebaran getaran-daya.
8. **Disiplin Pencegahan Data Leakage**: Partisi *train-test* sebelum transformasi skala.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path dataset telemetri mentah
data_path = Path("../code/data/telemetry_raw.csv")
df_raw = pd.read_csv(data_path)

print("=== RINGKASAN DATASET TELEMETRI MENTAH ===")
print(f"Dimensi Awal: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")
print("\nTipe Data Kolom:")
print(df_raw.dtypes)
print("\n5 Baris Pertama:")
print(df_raw.head())


=== RINGKASAN DATASET TELEMETRI MENTAH ===
Dimensi Awal: 24 baris x 8 kolom

Tipe Data Kolom:
device_id          object
timestamp          object
zone               object
temperature_c     float64
humidity_pct      float64
vibration_mms     float64
power_kw          float64
cooling_status     object
dtype: object

5 Baris Pertama:
  device_id            timestamp         zone  temperature_c  humidity_pct  vibration_mms  power_kw cooling_status
0   TURB-01  2026-03-01 08:00:00   Production           42.5          65.0           0.85     120.4         ACTIVE
1   TURB-01  2026-03-01 08:05:00   Production           43.1          64.5           0.88     122.1         ACTIVE
2   TURB-01  2026-03-01 08:10:00   production            NaN          64.0           0.90     121.8         ACTIVE
3   TURB-02  2026-03-01 08:00:00  Server Room           24.2          45.0           0.15      45.2       STANDBY
4   TURB-02  2026-03-01 08:05:00  server room           24.5          46.2           0.18   

## 2. Pembersihan Inkonsistensi String & Duplikasi Data
Pada data mentah di atas, kolom `zone` memiliki variasi penulisan seperti `'Production'` vs `'production'` dan `'Server Room'` vs `'server room'`. Kita lakukan standardisasi teks terlebih dahulu, kemudian mengidentifikasi duplikasi baris.


In [2]:
# Standardisasi teks zona
df_clean = df_raw.copy()
df_clean['zone'] = df_clean['zone'].astype(str).str.strip().str.title()
df_clean['cooling_status'] = df_clean['cooling_status'].astype(str).str.strip().str.upper()

# Periksa duplikasi identik
duplicate_rows = df_clean[df_clean.duplicated(keep=False)]
print(f"Jumlah baris duplikat identik: {df_clean.duplicated().sum()}")
if len(duplicate_rows) > 0:
    print(duplicate_rows[['device_id', 'timestamp', 'zone', 'temperature_c']])

# Menghapus duplikat dan mempertahankan entri pertama
df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
print(f"Dimensi setelah deduplikasi: {df_clean.shape}")


Jumlah baris duplikat identik: 1
   device_id            timestamp        zone  temperature_c
7    TURB-03  2026-03-01 08:05:00  Warehouse           31.2
13   TURB-03  2026-03-01 08:05:00  Warehouse           31.2
Dimensi setelah deduplikasi: (23, 8)


## 3. Analisis Missing Values & Imputasi Berbasis Grup
Kita mengecek kolom mana saja yang memiliki nilai kosong (*missing values*) dan menerapkan imputasi median berdasarkan kelompok zona (`zone`).


In [3]:
# Identifikasi Missing Values
missing_summary = df_clean.isnull().sum()
print("Jumlah Missing Values per Kolom:")
print(missing_summary[missing_summary > 0])

# Imputasi Group Median: nilai suhu dan kelembaban sangat bergantung pada karakteristik zona operasional
imputed_temps = df_clean.groupby('zone')['temperature_c'].transform(lambda s: s.fillna(s.median()))
imputed_humids = df_clean.groupby('zone')['humidity_pct'].transform(lambda s: s.fillna(s.median()))

df_clean['temperature_c'] = imputed_temps
df_clean['humidity_pct'] = imputed_humids

print("\nSisa Missing Values setelah Imputasi Terkelompok:")
print(df_clean[['temperature_c', 'humidity_pct']].isnull().sum())


Jumlah Missing Values per Kolom:
temperature_c    2
humidity_pct     1
dtype: int64

Sisa Missing Values setelah Imputasi Terkelompok:
temperature_c    0
humidity_pct     0
dtype: int64


## 4. Deteksi Outlier: Metode Tukey IQR vs Pemahaman Domain Engineering
Apakah nilai ekstrim selalu merupakan kesalahan sensor? Kita periksa anomali suhu dan getaran menggunakan kaidah IQR ($Q_1 - 1.5 \cdot IQR$ hingga $Q_3 + 1.5 \cdot IQR$).


In [4]:
# Deteksi Outlier Suhu dengan Tukey IQR
q1 = df_clean['temperature_c'].quantile(0.25)
q3 = df_clean['temperature_c'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df_clean[(df_clean['temperature_c'] < lower_bound) | (df_clean['temperature_c'] > upper_bound)]
print(f"Batas Bawah IQR: {lower_bound:.2f} °C, Batas Atas IQR: {upper_bound:.2f} °C")
print(f"Jumlah Outlier Terdeteksi: {len(outliers)}")
print(outliers[['device_id', 'timestamp', 'zone', 'temperature_c', 'vibration_mms', 'power_kw']])


Batas Bawah IQR: 7.95 °C, Batas Atas IQR: 57.15 °C
Jumlah Outlier Terdeteksi: 1
   device_id            timestamp        zone  temperature_c  vibration_mms  power_kw
10   TURB-01  2026-03-01 08:40:00  Production           88.5           3.85     210.5


**Wawasan Domain:**  
Observasi indeks 10 mencatat suhu $88.5^\circ\text{C}$ pada turbin `TURB-01`. Catatan ini diiringi lonjakan getaran mekanis ($3.85\text{ mm/s}$) dan konsumsi daya tinggi ($210.5\text{ kW}$). Ini **bukan glitch sensor**, melainkan insiden termal mekanis nyata (*overheating overload*). Menghapus baris ini secara buta akan menghilangkan sinyal kegagalan kritis yang vital bagi model deteksi anomali AI!


## 5. Transformasi Fitur Numerik: Normalisasi, Standarisasi, dan Log Transform
Kita amati kemiringan (*skewness*) distribusi `power_kw` sebelum dan sesudah transformasi logaritma, serta membandingkan Min-Max Scaling vs Standarisasi Z-score.


In [5]:
from scipy import stats

skew_raw = stats.skew(df_clean['power_kw'])
power_log = np.log1p(df_clean['power_kw'])
skew_log = stats.skew(power_log)

print(f"Skewness awal 'power_kw' : {skew_raw:+.4f} (Positively Skewed)")
print(f"Skewness pasca Log-Transform: {skew_log:+.4f} (Mendekati Simetris Normal)")

# Min-Max Scaling vs Standard Scaling
scaler_mm = MinMaxScaler()
scaler_std = StandardScaler()

df_clean['power_minmax'] = scaler_mm.fit_transform(df_clean[['power_kw']])
df_clean['power_standard'] = scaler_std.fit_transform(df_clean[['power_kw']])

print("\nPerbandingan Skala Fitur Daya (5 Sampel Pertama):")
print(df_clean[['power_kw', 'power_minmax', 'power_standard']].head())


Skewness awal 'power_kw' : +1.3959 (Positively Skewed)
Skewness pasca Log-Transform: +0.1829 (Mendekati Simetris Normal)

Perbandingan Skala Fitur Daya (5 Sampel Pertama):
   power_kw  power_minmax  power_standard
0     120.4      0.454930        0.395279
1     122.1      0.465215        0.428489
2     121.8      0.463399        0.422629
3      45.2      0.000000       -1.073801
4      46.0      0.004840       -1.058173


## 6. Encoding Fitur Kategorikal: Ordinal vs One-Hot Encoding


In [6]:
# One-Hot Encoding pada kolom Zone
df_encoded = pd.get_dummies(df_clean, columns=['zone', 'cooling_status'], drop_first=False, dtype=int)
encoded_cols = [c for c in df_encoded.columns if 'zone_' in c or 'cooling_status_' in c]

print("Kolom hasil One-Hot Encoding:")
print(df_encoded[encoded_cols].head())


Kolom hasil One-Hot Encoding:
   zone_Production  zone_Server Room  zone_Warehouse  cooling_status_ACTIVE  cooling_status_OFF  cooling_status_STANDBY
0                1                 0               0                      1                   0                       0
1                1                 0               0                      1                   0                       0
2                1                 0               0                      1                   0                       0
3                0                 1               0                      0                   0                       1
4                0                 1               0                      0                   0                       1


## 7. Analisis Korelasi Pearson
Korelasi linear Pearson mengukur derajat asosiasi antar sepasang fitur numerik dalam rentang $[-1, +1]$.


In [7]:
num_cols = ['temperature_c', 'humidity_pct', 'vibration_mms', 'power_kw']
corr_matrix = df_clean[num_cols].corr()

print("Matriks Korelasi Pearson:")
print(corr_matrix.round(4))


Matriks Korelasi Pearson:
               temperature_c  humidity_pct  vibration_mms  power_kw
temperature_c         1.0000        0.4852         0.8412    0.7845
humidity_pct          0.4852        1.0000         0.3120    0.3458
vibration_mms         0.8412        0.3120         1.0000    0.8921
power_kw              0.7845        0.3458         0.8921    1.0000


Terdapat korelasi positif yang sangat kuat antara **Getaran (*vibration_mms*)** dan **Daya Listrik (*power_kw*)** ($r = +0.8921$), serta antara **Suhu (*temperature_c*)** dan **Getaran** ($r = +0.8412$). Ini mengindikasikan beban mekanis turbin secara langsung memicu kenaikan suhu dan amplitudo getaran.


## 8. Disiplin Pencegahan Data Leakage
Mencegah kebocoran data (*data leakage*) adalah salah satu aturan emas dalam machine learning. Parameter statistik (seperti rata-rata $\mu$ dan simpangan baku $\sigma$) **hanya boleh dihitung dari subset data latih (*training split*)**.


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_clean[['temperature_c', 'humidity_pct', 'vibration_mms', 'power_kw']].values
y = (df_clean['temperature_c'] > 40.0).astype(int).values

# 1. Partisi data terlebih dahulu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Fit HANYA pada data latih, lalu transform kedua split
strict_scaler = StandardScaler()
X_train_scaled = strict_scaler.fit_transform(X_train)
X_test_scaled = strict_scaler.transform(X_test)

# Bandingkan dengan skenario bocor (fit pada seluruh populasi X)
leaky_scaler = StandardScaler()
leaky_scaler.fit(X)

mean_train_strict = strict_scaler.mean_[0]
mean_leaky = leaky_scaler.mean_[0]
diff_bias = abs(mean_train_strict - mean_leaky)

print(f"Mean Suhu (Strict Training): {mean_train_strict:.4f} °C")
print(f"Mean Suhu (Leaky Populer)  : {mean_leaky:.4f} °C")
print(f"Bias Kebocoran Data        : {diff_bias:.4f} °C")
print("Status Pipeline: Valid & Aman dari Data Leakage.")


Mean Suhu (Strict Training): 34.6250 °C
Mean Suhu (Leaky Populer)  : 35.3417 °C
Bias Kebocoran Data        : 0.7167 °C
Status Pipeline: Valid & Aman dari Data Leakage.
